# 18.10 蒸餾是否幫到學生？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**學生很像教師，但教師本身錯，是否算成功？

同一學生架構、初始化與資料預算比較純SFT和蒸餾。教師額外forward或生成不是免費，另列時間與tokens成本。

**把直覺寫成數學：**matched student與budget；評價使用獨立真值，不能拿teacher一致率替代正確率。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
plans = [
    {"method": "SFT", "student_width": 8, "steps": 100, "teacher_forward": False},
    {"method": "KL", "student_width": 8, "steps": 100, "teacher_forward": True},
]
print(plans)
print("入口：train.py --task distill --teacher checkpoints/teacher.pt --train --width 8")

**如何讀結果：**這裏只給可執行計劃，未聲稱蒸餾提升；無teacher時CLI正式訓練會拒絕繼續。

**只改一件事：**只改teacher的品質，保持學生配置。
